# GroupProject — Smart Invigilation (improved exam-cheating classifier)

**Group Project**

Separate, upgraded training notebook vs `Exam_Cheating_TPU_Training.ipynb`. Same Drive datasets and backend export format (`best.pt`), aimed at **higher OOF macro-F1 / AP** and stabler live flagging.

### What is improved over the folder baseline
1. **Stronger backbone** — EfficientNet-B3 (falls back to B0 if OOM) instead of B0-only
2. **EMA weights** — exponential moving average used for validation & export
3. **MixUp** — soft labels reduce overconfidence on noisy exam crops
4. **Test-time augmentation (TTA)** — horizontal flip averaging for OOF thresholding
5. **Fold ensemble export** — average of fold EMAs (not only one full-data retrain)
6. **Warmup + cosine + grad clip** — stabler fine-tune on TPU/GPU
7. **Temperature calibration** on OOF probabilities before threshold search
8. **Soft labels** for ambiguous `frontwatching` (configurable)
9. Works on **Colab TPU / CUDA / CPU**; `SMOKE_TEST=1` for a local box check

**Runtime → Change runtime type → TPU (v5e-1) or GPU.** Cite datasets + declare AI assistance in your report.


Install extras (Torch / TPU already on Colab).


In [ ]:
import os, sys, subprocess
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
AUTHOR = "GroupProject"
if not SMOKE_TEST:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "imagehash", "kaggle", "lap"], check=True)
print("Author / file:", AUTHOR)


Author / file: B226715A


## Settings — edit this cell
Point `DRIVE_ROOT` at the shared SmartInvigilation folder. Keep source paths aligned with the Drive layout.


In [ ]:
CFG = dict(
    AUTHOR="GroupProject",
    SOURCE="drive",
    DRIVE_ROOT="/content/drive/MyDrive/SmartInvigilation",
    SOURCES=[
        dict(name="examdatasetnew", type="images", path="examdatasetnew/datasetnew"),
        dict(name="ExamCheatingDataset", type="images", path="archive(1)/ExamCheatingDataset/train",
             kaggle="ardutraagiginting/exam-cheating-dataset"),
        dict(name="hashemite", type="video", path="hashemite", enabled=False, fps=1.0,
             kaggle="fairouzfarouq/student-cheating-behavior-dataset",
             label_map={"not cheating": 0, "normal": 0, "exchang": 1, "looking": 1, "sheet": 1, "phone": 1, "mobile": 1}),
    ],
    OUT_DIR="/content/drive/MyDrive/SmartInvigilation_runs/GroupProject_effnetb3",
    WORK_DIR="/content/work_GroupProject",

    NORMAL_CLASSES=["normal", "normal act", "not cheating", "notcheating"],
    # soft label in [0,1] for ambiguous frontwatching (baseline used hard 0/1)
    FRONTWATCHING_SOFT=0.35,
    CROSS_PAIRS=[("examdatasetnew", "ExamCheatingDataset")],

    ARCH="efficientnet_b3",       # efficientnet_b3 | efficientnet_b0 | resnet50
    ARCH_FALLBACK="efficientnet_b0",
    HEAD="binary",
    IMG=288,                      # B3 likes 288; B0 fallback uses 224 if OOM path triggers
    CACHE_SIZE=320,
    BATCH=32, EPOCHS_HEAD=4, EPOCHS_FT=30, LR_HEAD=1e-3, LR_FT=8e-5, WD=1e-4,
    WARMUP_EPOCHS=2, GRAD_CLIP=1.0,
    PATIENCE=6, LOSS="weighted_ce",
    BF16=True,
    N_FOLDS=5, FOLDS_TO_RUN=5,
    TARGET_PRECISION=0.85,        # stricter than baseline 0.80
    EMA_DECAY=0.995,
    MIXUP_ALPHA=0.2,
    USE_TTA=True,
    EXPORT_ENSEMBLE=True,

    DROP_FLIP_COPY_FILES=True, DEDUP_HAMMING=4,
    YOLO_WEIGHTS="yolov8n.pt", YOLO_CONF=0.3, CROP_IF_PERSON_AREA_BELOW=0.6, CROP_PAD=0.12, MIN_CROP_PX=40,
    SEED=42,
)
if SMOKE_TEST:
    import tempfile
    _smoke = os.path.join(tempfile.gettempdir(), "GroupProject_smoke")
    CFG.update(SOURCE="synthetic", DRIVE_ROOT=os.path.join(_smoke, "data"), OUT_DIR=os.path.join(_smoke, "out"),
               WORK_DIR=os.path.join(_smoke, "work"),
               SOURCES=[dict(name="examdatasetnew", type="images", path="examdatasetnew"),
                        dict(name="ExamCheatingDataset", type="images", path="ExamCheatingDataset")],
               ARCH="efficientnet_b0", ARCH_FALLBACK="efficientnet_b0",
               BATCH=8, EPOCHS_HEAD=1, EPOCHS_FT=2, N_FOLDS=3, FOLDS_TO_RUN=2, PATIENCE=1,
               IMG=64, CACHE_SIZE=72, MIXUP_ALPHA=0.1, USE_TTA=True, EXPORT_ENSEMBLE=True,
               TARGET_PRECISION=0.5, WARMUP_EPOCHS=0)
SOURCES = [s for s in CFG["SOURCES"] if s.get("enabled", True)]
print(CFG["AUTHOR"], "| arch", CFG["ARCH"], "| img", CFG["IMG"], "| sources", [s["name"] for s in SOURCES])


B226715A | arch efficientnet_b3 | img 288 | sources ['examdatasetnew', 'ExamCheatingDataset']


Device selection (TorchTPU → XLA → CUDA → CPU).


In [ ]:
import contextlib, torch, copy, math
BACKEND, xm = "cpu", None
DEVICE = torch.device("cpu")
try:
    import torch_tpu  # noqa: F401
    DEVICE, BACKEND = torch.device("tpu"), "torch_tpu"
except Exception:
    try:
        import torch_xla, torch_xla.core.xla_model as xm
        DEVICE = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
        BACKEND = "torch_xla"
    except Exception:
        if torch.cuda.is_available():
            DEVICE, BACKEND = torch.device("cuda"), "cuda"
if SMOKE_TEST:
    DEVICE, BACKEND = torch.device("cpu"), "cpu"

def autocast():
    if not CFG["BF16"] or BACKEND == "cpu":
        return contextlib.nullcontext()
    if BACKEND == "cuda" and not torch.cuda.is_bf16_supported():
        return contextlib.nullcontext()
    kind = {"torch_tpu": "tpu", "torch_xla": "xla", "cuda": "cuda"}[BACKEND]
    return torch.autocast(device_type=kind, dtype=torch.bfloat16)

def sync():
    if BACKEND == "torch_xla":
        (torch_xla.sync if hasattr(torch_xla, "sync") else xm.mark_step)()

def opt_step(opt):
    if BACKEND == "torch_xla":
        xm.optimizer_step(opt)
    else:
        opt.step()
    sync()

try:
    with autocast():
        (torch.ones(2, 2, device=DEVICE) @ torch.ones(2, 2, device=DEVICE)).cpu()
except Exception as e:
    print("bf16 disabled:", e); CFG["BF16"] = False
print("backend:", BACKEND, "| device:", DEVICE, "| torch", torch.__version__)


backend: torch_tpu | device: tpu | torch 2.14.1+cpu


Mount Drive / optional Kaggle download.


In [ ]:
from pathlib import Path
if not SMOKE_TEST and (CFG["SOURCE"] == "drive" or str(CFG["OUT_DIR"]).startswith("/content/drive")):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped:", e)

def kaggle_download(slug, dest):
    from google.colab import files
    kj = Path.home() / ".kaggle/kaggle.json"
    if not kj.exists():
        print("Upload kaggle.json")
        up = files.upload(); kj.parent.mkdir(exist_ok=True)
        kj.write_bytes(up["kaggle.json"]); kj.chmod(0o600)
    subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", dest, "--unzip"], check=True)

def src_root(s):
    p = Path(s["path"]) if Path(s["path"]).is_absolute() else Path(CFG["DRIVE_ROOT"]) / s["path"]
    if (CFG["SOURCE"] == "kaggle" or not p.exists()) and s.get("kaggle") and not SMOKE_TEST:
        dest = Path("/content/data") / s["name"]
        if not dest.exists(): kaggle_download(s["kaggle"], str(dest))
        p = dest
    return p
for d in (CFG["OUT_DIR"], CFG["WORK_DIR"]):
    Path(d).mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


SMOKE_TEST synthetic data.


In [ ]:
if SMOKE_TEST:
    import numpy as np, cv2
    from PIL import Image
    try:
        from ultralytics.utils import ASSETS
        people = [np.array(Image.open(ASSETS / f).convert("RGB")) for f in ("zidane.jpg", "bus.jpg")]
    except Exception:
        people = [(np.random.rand(120, 90, 3) * 255).astype("uint8") for _ in range(2)]
    rng = np.random.default_rng(0)
    layout = {"examdatasetnew": ["backwatching", "frontwatching", "normal", "showinggestures", "sidewatching", "suspiciouswatching"],
              "ExamCheatingDataset": ["normal act", "looking friend", "giving code", "giving object", "cheating"]}
    for ds, classes in layout.items():
        for ci, c in enumerate(classes):
            d = Path(CFG["DRIVE_ROOT"]) / ds / c; d.mkdir(parents=True, exist_ok=True)
            for i in range(14):
                a = (rng.random((96, 80, 3)) * 60).astype("uint8"); a[:, :, ci % 3] += 120 + 10 * ci
                Image.fromarray(a).save(d / f"{i}.jpg")
            Image.fromarray(a).save(d / f"{i}_f.jpg")
            for k in range(2):
                b = people[k % 2].astype(int) + rng.integers(-15, 15, people[k % 2].shape)
                Image.fromarray(np.clip(b, 0, 255).astype("uint8")).save(d / f"group_{k}.jpg")
    print("synthetic data ready")


Collect images/videos into one table (soft label for frontwatching).


In [ ]:
import re, numpy as np, pandas as pd, cv2
IMG_EXT, VID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}, {".mp4", ".avi", ".mov", ".mkv", ".webm"}
norm = lambda s: re.sub(r"[_\-]+", " ", str(s).strip().lower())

def hard_cheating(cls):
    c = norm(cls)
    if c in [norm(x) for x in CFG["NORMAL_CLASSES"]]: return 0
    if c == "frontwatching": return None
    return 1

def label_of(s, cls, rel):
    lm = {norm(k): v for k, v in (s.get("label_map") or {}).items()}
    if lm:
        if norm(cls) in lm: return float(lm[norm(cls)])
        hit = next((v for k, v in lm.items() if k in norm(rel)), None)
        return None if hit is None else float(hit)
    if norm(cls) == "frontwatching":
        return float(CFG["FRONTWATCHING_SOFT"])
    y = hard_cheating(cls)
    return None if y is None else float(y)

rows, skipped = [], []
for s in SOURCES:
    root = src_root(s)
    if not root.exists(): print("MISSING:", s["name"], root); continue
    kind, multi = s.get("type", "images"), s.get("multi", "all" if s.get("type") == "video" else "largest")
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root); cls = norm(rel.parts[0]) if len(rel.parts) > 1 else norm(p.stem)
        if kind == "images" and p.suffix.lower() in IMG_EXT:
            y = label_of(s, cls, rel)
            if y is None: skipped.append(str(rel)); continue
            rows.append(dict(path=str(p), dataset=s["name"], kind="images", cls=cls, y=y, y_hard=int(y >= 0.5),
                             subject=None, multi=multi, train=s.get("train", True)))
        elif kind == "video" and p.suffix.lower() in VID_EXT:
            y = label_of(s, cls, rel)
            if y is None: skipped.append(str(rel)); continue
            fdir = Path(CFG["WORK_DIR"]) / "frames" / s["name"]; fdir.mkdir(parents=True, exist_ok=True)
            cap = cv2.VideoCapture(str(p)); fps = cap.get(cv2.CAP_PROP_FPS) or 25
            step, i, vid = max(int(round(fps / s.get("fps", 1.0))), 1), 0, norm(str(rel.with_suffix(""))).replace("/", "_")
            while True:
                ok, fr = cap.read()
                if not ok: break
                if i % step == 0:
                    out = fdir / f"{vid}_{i:06d}.jpg"; cv2.imwrite(str(out), fr)
                    rows.append(dict(path=str(out), dataset=s["name"], kind="video", cls=cls, y=y, y_hard=int(y >= 0.5),
                                     subject=f"{s['name']}/{vid}", multi=multi, train=s.get("train", True)))
                i += 1
            cap.release()
df = pd.DataFrame(rows)
if skipped: print(f"no label for {len(skipped)} files, e.g.", skipped[:5])
print(df.groupby(["dataset", "cls"]).agg(n=("y", "size"), y_mean=("y", "mean")).to_string())


MISSING: examdatasetnew /content/drive/MyDrive/SmartInvigilation/examdatasetnew/datasetnew
Upload kaggle.json


if CFG["DROP_FLIP_COPY_FILES"]:
    stem = df.path.map(lambda p: Path(p).stem.lower())
    bad = (df.kind == "images") & (stem.str.contains(r"copy") | stem.str.contains(r"_f(?:\s*\(\d+\))?$"))
    print("dropped flip/copy:", int(bad.sum()))
    df = df[~bad].reset_index(drop=True)

In [ ]:
if CFG["DROP_FLIP_COPY_FILES"]:
    stem = df.path.map(lambda p: Path(p).stem.lower())
    bad = (df.kind == "images") & (stem.str.contains(r"copy") | stem.str.contains(r"_f(?:\s*\(\d+\))?$"))
    print("dropped flip/copy:", int(bad.sum())); df = df[~bad].reset_index(drop=True)


import imagehash
from PIL import Image, ImageOps

def phashes(p):
    im = Image.open(p).convert("L")
    return int(str(imagehash.phash(im)), 16), int(str(imagehash.phash(ImageOps.mirror(im))), 16)

im_idx = np.nonzero((df.kind == "images").values)[0]
if len(im_idx):
    H = np.array([phashes(df.path[i]) for i in im_idx], dtype=np.uint64).reshape(-1, 2)
    h, hm = H[:, 0], H[:, 1]
    popcnt = (lambda a: np.bitwise_count(a)) if hasattr(np, "bitwise_count") else (lambda a: np.unpackbits(a.view(np.uint8).reshape(*a.shape, 8), -1).sum(-1))
    parent = list(range(len(im_idx)))
    def find(i):
        while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(len(im_idx)):
        d = np.minimum(popcnt(h[i] ^ h[i + 1:]), popcnt(h[i] ^ hm[i + 1:]))
        for j in np.nonzero(d <= CFG["DEDUP_HAMMING"])[0] + i + 1:
            parent[find(int(j))] = find(i)
    df.loc[im_idx, "subject"] = [f"img/g{find(k)}" for k in range(len(im_idx))]
print("rows:", len(df), "| image groups:", df.loc[im_idx, "subject"].nunique() if len(im_idx) else 0)

In [ ]:
import imagehash
from PIL import Image, ImageOps
def phashes(p):
    im = Image.open(p).convert("L")
    return int(str(imagehash.phash(im)), 16), int(str(imagehash.phash(ImageOps.mirror(im))), 16)
im_idx = np.nonzero((df.kind == "images").values)[0]
if len(im_idx):
    H = np.array([phashes(df.path[i]) for i in im_idx], dtype=np.uint64).reshape(-1, 2)
    h, hm = H[:, 0], H[:, 1]
    popcnt = (lambda a: np.bitwise_count(a)) if hasattr(np, "bitwise_count") else (lambda a: np.unpackbits(a.view(np.uint8).reshape(*a.shape, 8), -1).sum(-1))
    parent = list(range(len(im_idx)))
    def find(i):
        while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(len(im_idx)):
        d = np.minimum(popcnt(h[i] ^ h[i + 1:]), popcnt(h[i] ^ hm[i + 1:]))
        for j in np.nonzero(d <= CFG["DEDUP_HAMMING"])[0] + i + 1:
            parent[find(int(j))] = find(i)
    df.loc[im_idx, "subject"] = [f"img/g{find(k)}" for k in range(len(im_idx))]
print("rows:", len(df), "| image groups:", df.loc[im_idx, "subject"].nunique() if len(im_idx) else 0)


YOLO person crops (cached).


In [ ]:
import subprocess, sys
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "imagehash"], check=True)

from ultralytics import YOLO
from PIL import Image
from pathlib import Path
import numpy as np
import pandas as pd

yolo = YOLO(CFG["YOLO_WEIGHTS"])
cache = Path(CFG["WORK_DIR"]) / f"crops{CFG['CACHE_SIZE']}"
cache.mkdir(parents=True, exist_ok=True)
S, out_rows, n_multi = CFG["CACHE_SIZE"], [], 0

for r in df.itertuples():
    im = Image.open(r.path).convert("RGB")
    W, Hh = im.size
    res = yolo.predict(im, classes=[0], conf=CFG["YOLO_CONF"], device="cpu", verbose=False)[0]
    b = res.boxes.xyxy.cpu().numpy() if len(res.boxes) else np.zeros((0, 4))
    b = b[np.minimum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]) >= CFG["MIN_CROP_PX"]]
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    if len(b) > 1:
        n_multi += 1
        b = b if r.multi == "all" else b[[area.argmax()]]
    elif len(b) == 1 and r.kind == "images" and area[0] / (W * Hh) >= CFG["CROP_IF_PERSON_AREA_BELOW"]:
        b = np.zeros((0, 4))
    crops = []
    for x1, y1, x2, y2 in b:
        px, py = CFG["CROP_PAD"] * (x2 - x1), CFG["CROP_PAD"] * (y2 - y1)
        crops.append(im.crop((max(0, x1 - px), max(0, y1 - py), min(W, x2 + px), min(Hh, y2 + py))))
    if not crops and r.kind == "images":
        crops = [im]
    for k, c in enumerate(crops):
        out = cache / f"{r.Index:06d}_{k}.jpg"
        c.resize((S, S), Image.BILINEAR).save(out, quality=95)
        out_rows.append({**r._asdict(), "crop": str(out)})

data = pd.DataFrame(out_rows).drop(columns=["Index"], errors="ignore")
print("multi-person:", n_multi, "| crops:", len(data))
print(data.groupby(["dataset", "y_hard"]).size().unstack(fill_value=0))

Stratified grouped folds.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
import numpy as np

CLASS_NAMES, CHEAT_CLASSES = ["normal", "cheating"], ["cheating"]
data["t"] = data.y_hard
CHEAT_IDX = [1]
train_all = data[data.train].reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
train_all["fold"] = -1
_groups = train_all.subject.where(train_all.subject.notna(), train_all.index.map(lambda i: f"row{i}"))
for f, (_, va) in enumerate(sgkf.split(train_all, train_all.t, groups=_groups)):
    train_all.loc[va, "fold"] = f
print(CLASS_NAMES)
print(pd.crosstab(train_all.fold, [train_all.dataset, train_all.y_hard]))

Augmentations + MixUp.


In [ ]:
import io, random
import torchvision.transforms as T
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
class DownUp:
    def __call__(self, im):
        if random.random() > 0.25: return im
        s = random.randint(max(16, CFG["IMG"] // 4), max(17, CFG["IMG"] // 2))
        return im.resize((s, s), Image.BILINEAR).resize(im.size, Image.BILINEAR)
class Jpeg:
    def __call__(self, im):
        if random.random() > 0.3: return im
        b = io.BytesIO(); im.save(b, "JPEG", quality=random.randint(28, 92)); b.seek(0)
        return Image.open(b).convert("RGB")
train_tf = T.Compose([
    T.RandomResizedCrop(CFG["IMG"], scale=(0.55, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.RandomRotation(12)], p=0.5),
    T.ColorJitter(0.35, 0.35, 0.25, 0.04),
    T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.25),
    T.RandomAutocontrast(p=0.2),
    DownUp(), Jpeg(),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((CFG["IMG"], CFG["IMG"])), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf_flip = T.Compose([T.Resize((CFG["IMG"], CFG["IMG"])), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(MEAN, STD)])

def mixup_batch(x, y, alpha):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


Dataset / loaders / TTA predict.


In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
class Crops(Dataset):
    def __init__(self, d, tf, soft=False):
        self.p = d.crop.tolist()
        self.t = (d.y.tolist() if soft else d.t.tolist())
        self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), self.t[i]

NW = 0 if SMOKE_TEST or os.name == "nt" else min(8, os.cpu_count() or 2)
def train_loader(d):
    ds = Crops(d, train_tf, soft=False)
    if CFG["LOSS"] == "sampler":
        w = 1.0 / d.t.map(d.t.value_counts()).values
        return DataLoader(ds, CFG["BATCH"], sampler=WeightedRandomSampler(w, len(d), True), drop_last=True, num_workers=NW)
    return DataLoader(ds, CFG["BATCH"], shuffle=True, drop_last=True, num_workers=NW)

@torch.no_grad()
def _predict_tf(model, d, tf):
    model.eval(); out = []
    for x, _ in DataLoader(Crops(d, tf), CFG["BATCH"], num_workers=NW):
        n = len(x)
        if n < CFG["BATCH"]: x = torch.cat([x, x.new_zeros(CFG["BATCH"] - n, *x.shape[1:])])
        with autocast():
            p = torch.softmax(model(x.to(DEVICE)).float(), 1)[:, CHEAT_IDX].sum(1)
        sync(); out.append(p.cpu()[:n])
    return torch.cat(out).numpy()

@torch.no_grad()
def predict(model, d, tta=None):
    use_tta = CFG["USE_TTA"] if tta is None else tta
    p0 = _predict_tf(model, d, eval_tf)
    if not use_tta: return p0
    p1 = _predict_tf(model, d, eval_tf_flip)
    return 0.5 * (p0 + p1)


Model builder + EMA helper.


In [ ]:
import torch.nn as nn
from torchvision import models

def build_model(n, arch=None):
    arch = arch or CFG["ARCH"]
    try:
        if arch == "resnet50":
            m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n); head = m.fc
        elif arch == "efficientnet_b3":
            m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, n); head = m.classifier
        else:
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, n); head = m.classifier
        return m, head, arch
    except Exception as e:
        print("arch", arch, "failed:", e, "→ fallback", CFG["ARCH_FALLBACK"])
        CFG["ARCH"] = CFG["ARCH_FALLBACK"]; CFG["IMG"] = min(CFG["IMG"], 224)
        return build_model(n, CFG["ARCH_FALLBACK"])

class ModelEMA:
    def __init__(self, model, decay):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
        self.decay = decay
    @torch.no_grad()
    def update(self, model):
        d = self.decay
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(msd[k])


Metrics + temperature calibration + threshold.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix, precision_recall_curve

def metrics(y, p, thr=0.5):
    yh = (p >= thr).astype(int)
    return dict(threshold=float(thr), precision=float(precision_score(y, yh, zero_division=0)),
                recall=float(recall_score(y, yh, zero_division=0)),
                macro_f1=float(f1_score(y, yh, average="macro", zero_division=0)),
                ap=float(average_precision_score(y, p)) if len(set(y)) > 1 else float("nan"),
                confusion=confusion_matrix(y, yh, labels=[0, 1]).tolist(), n=int(len(y)))

def fit_temperature(y, logits_or_p, grid=np.linspace(0.5, 3.0, 26)):
    p = np.clip(logits_or_p, 1e-6, 1 - 1e-6)
    z = np.log(p / (1 - p))
    best_t, best_ap = 1.0, -1.0
    for t in grid:
        pt = 1 / (1 + np.exp(-z / t))
        ap = average_precision_score(y, pt) if len(set(y)) > 1 else 0
        if ap > best_ap: best_ap, best_t = ap, float(t)
    return best_t

def apply_temperature(p, t):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    z = np.log(p / (1 - p))
    return 1 / (1 + np.exp(-z / t))

def threshold_for_precision(y, p, target):
    pr, rc, th = precision_recall_curve(y, p)
    ok = np.nonzero(pr[:-1] >= target)[0]
    return float(th[ok[np.argmax(rc[ok])]]) if len(ok) else 0.5


Two-stage train with MixUp, warmup, EMA, grad clip.


In [ ]:
import time

def train(tr, va=None, epochs_ft=None):
    torch.manual_seed(CFG["SEED"]); random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
    model, head, used_arch = build_model(len(CLASS_NAMES)); model.to(DEVICE)
    CFG["ARCH"] = used_arch
    ema = ModelEMA(model, CFG["EMA_DECAY"])
    cw = None
    if CFG["LOSS"] == "weighted_ce":
        cnt = np.bincount(tr.t, minlength=len(CLASS_NAMES)).astype(float)
        cw = torch.tensor(cnt.sum() / np.maximum(cnt, 1) / len(cnt), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
    dl = train_loader(tr)
    best, best_state, best_ep, bad, hist = -1, None, 0, 0, []
    stages = [("head", CFG["EPOCHS_HEAD"], CFG["LR_HEAD"]), ("ft", epochs_ft or CFG["EPOCHS_FT"], CFG["LR_FT"])]
    for stage, n_ep, lr in stages:
        for p in model.parameters(): p.requires_grad = (stage == "ft")
        for p in head.parameters(): p.requires_grad = True
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=CFG["WD"])
        steps = max(n_ep * len(dl), 1)
        warm = CFG["WARMUP_EPOCHS"] * len(dl) if stage == "ft" else 0
        def lr_at(step):
            if step < warm:
                return lr * (step + 1) / max(warm, 1)
            prog = (step - warm) / max(steps - warm, 1)
            return lr * 0.5 * (1 + math.cos(math.pi * prog))
        step_i = 0
        for ep in range(n_ep):
            model.train(); t0 = time.time(); tot = 0.0
            for x, t in dl:
                x, t = x.to(DEVICE), t.to(DEVICE)
                for g in opt.param_groups: g["lr"] = lr_at(step_i)
                x_m, y_a, y_b, lam = mixup_batch(x, t, CFG["MIXUP_ALPHA"] if stage == "ft" else 0.0)
                with autocast():
                    logits = model(x_m).float()
                    if CFG["MIXUP_ALPHA"] > 0 and stage == "ft":
                        loss = lam * loss_fn(logits, y_a) + (1 - lam) * loss_fn(logits, y_b)
                    else:
                        loss = loss_fn(logits, t)
                opt.zero_grad(); loss.backward()
                if CFG["GRAD_CLIP"]:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                opt_step(opt); ema.update(model); step_i += 1
                tot += float(loss.detach().cpu())
            rec = dict(stage=stage, epoch=ep, loss=tot / max(len(dl), 1), sec=round(time.time() - t0, 1))
            if va is not None and stage == "ft":
                rec["val_macro_f1"] = metrics(va.y_hard.values, predict(ema.ema, va))["macro_f1"]
                if rec["val_macro_f1"] > best:
                    best, best_ep, bad = rec["val_macro_f1"], ep + 1, 0
                    best_state = {k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()}
                else:
                    bad += 1
            print(rec); hist.append(rec)
            if va is not None and stage == "ft" and bad >= CFG["PATIENCE"]: break
    if best_state is not None:
        model.load_state_dict(best_state); ema.ema.load_state_dict(best_state)
    return ema.ema, best_ep or (epochs_ft or CFG["EPOCHS_FT"]), hist


Grouped CV with TTA + temperature calibration.


In [ ]:
oof = np.full(len(train_all), np.nan); fold_res, best_eps, fold_states = [], [], []
for f in range(min(CFG["FOLDS_TO_RUN"], CFG["N_FOLDS"])):
    tr, va = train_all[train_all.fold != f], train_all[train_all.fold == f]
    print(f"--- fold {f}: train {len(tr)}  val {len(va)}")
    model, ep, hist = train(tr, va)
    oof[va.index] = predict(model, va)
    best_eps.append(ep)
    fold_states.append({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    fold_res.append(dict(fold=f, best_epoch=ep, **metrics(va.y_hard.values, oof[va.index])))
m = ~np.isnan(oof)
TEMP = fit_temperature(train_all.y_hard.values[m], oof[m])
oof_cal = apply_temperature(oof[m], TEMP)
THRESH = threshold_for_precision(train_all.y_hard.values[m], oof_cal, CFG["TARGET_PRECISION"])
cv = dict(temperature=TEMP,
          at_0_5=metrics(train_all.y_hard.values[m], oof[m]),
          at_calibrated=metrics(train_all.y_hard.values[m], oof_cal, THRESH))
print("OOF raw @0.5:", cv["at_0_5"])
print(f"OOF calibrated T={TEMP:.2f} thr={THRESH:.3f}:", cv["at_calibrated"])
train_all.assign(p_cheat=oof).to_csv(f"{CFG['OUT_DIR']}/oof_predictions_{CFG['AUTHOR']}.csv", index=False)


Cross-dataset generalization check.


In [ ]:
cross = {}
present = set(data.dataset)
for a, b in CFG["CROSS_PAIRS"]:
    if a not in present or b not in present: print("skip", a, b); continue
    for A, B in [(a, b), (b, a)]:
        tr, te = data[data.dataset == A], data[data.dataset == B]
        model, _, _ = train(tr.reset_index(drop=True), epochs_ft=int(np.median(best_eps) if best_eps else CFG["EPOCHS_FT"]))
        p = apply_temperature(predict(model, te.reset_index(drop=True)), TEMP)
        cross[f"{A}->{B}"] = dict(at_0_5=metrics(te.y_hard.values, p), at_cv_threshold=metrics(te.y_hard.values, p, THRESH))
        print(A, "->", B, cross[f"{A}->{B}"]["at_0_5"])


Export `best.pt` (ensemble of fold EMAs) + metrics for GroupProject.


In [ ]:
import json

def average_state_dicts(states):
    out = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in states[0].items() if v.dtype.is_floating_point}
    other = {k: v for k, v in states[0].items() if not v.dtype.is_floating_point}
    for st in states:
        for k in out: out[k] += st[k].float()
    for k in out: out[k] /= len(states)
    out.update(other)
    return out

if CFG["EXPORT_ENSEMBLE"] and fold_states:
    ens = average_state_dicts(fold_states)
    final, _, used = build_model(len(CLASS_NAMES))
    final.load_state_dict(ens, strict=False)
    export_model = final
    export_note = f"fold_ema_ensemble_n={len(fold_states)}"
else:
    export_model, _, _ = train(train_all, epochs_ft=int(np.median(best_eps) if best_eps else CFG["EPOCHS_FT"]))
    export_note = "full_data_retrain"

ckpt = {
    "author": CFG["AUTHOR"],
    "arch": CFG["ARCH"],
    "model_state_dict": {k: v.detach().cpu() for k, v in export_model.state_dict().items()},
    "class_names": CLASS_NAMES,
    "cheating_classes": CHEAT_CLASSES,
    "head": CFG["HEAD"],
    "input_size": CFG["IMG"],
    "mean": MEAN,
    "std": STD,
    "threshold": THRESH,
    "temperature": TEMP,
    "export": export_note,
    "cfg": CFG,
}
torch.save(ckpt, f"{CFG['OUT_DIR']}/best.pt")
torch.save(ckpt, f"{CFG['OUT_DIR']}/{CFG['AUTHOR']}_best.pt")
metrics_blob = dict(author=CFG["AUTHOR"], backend=BACKEND, cv_folds=fold_res, cv=cv, cross_dataset=cross,
                    threshold=THRESH, temperature=TEMP, class_names=CLASS_NAMES,
                    cheating_classes=CHEAT_CLASSES, export=export_note, config=CFG)
json.dump(metrics_blob, open(f"{CFG['OUT_DIR']}/metrics_{CFG['AUTHOR']}.json", "w"), indent=2, default=str)
snippet = '# GroupProject improved classifier export\n' + f'''classifier:
  model_path: models/{CFG['AUTHOR']}_best.pt
  arch: {CFG['ARCH']}
  input_size: {CFG['IMG']}
  class_names: {json.dumps(CLASS_NAMES)}
  cheating_classes: {json.dumps(CHEAT_CLASSES)}
  temperature: {TEMP:.4f}
# flagging.threshold: start near {THRESH:.2f} (target precision {CFG['TARGET_PRECISION']})
'''
open(f"{CFG['OUT_DIR']}/config_snippet_{CFG['AUTHOR']}.yaml", "w").write(snippet)
print("saved", CFG["OUT_DIR"])
print(snippet)
print("CV summary:", cv["at_calibrated"])


## Try-it (image / video)
Loads the **GroupProject** checkpoint and applies temperature scaling before flagging.


In [ ]:
TRY_PATH = ""  #@param {type:"string"}
TRY_MODEL = f"{CFG['OUT_DIR']}/{CFG['AUTHOR']}_best.pt"
TRY_THRESHOLD = 0.0  #@param {type:"number"}
TRY_OUT = Path(os.path.join(CFG["OUT_DIR"], "try")); TRY_OUT.mkdir(parents=True, exist_ok=True)

import torchvision.transforms.functional as TF

def make_arch(arch, n, pretrained=False):
    if arch == "resnet50":
        m = models.resnet50(weights="IMAGENET1K_V2" if pretrained else None); m.fc = nn.Linear(m.fc.in_features, n)
    elif arch == "efficientnet_b3":
        m = models.efficientnet_b3(weights="IMAGENET1K_V1" if pretrained else None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    else:
        m = models.efficientnet_b0(weights="IMAGENET1K_V1" if pretrained else None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    return m

ck = torch.load(TRY_MODEL, map_location="cpu", weights_only=False)
n = [v for k, v in ck["model_state_dict"].items() if k.endswith("weight") and v.ndim == 2][-1].shape[0]
tmodel = make_arch(ck.get("arch", "efficientnet_b3"), n)
tmodel.load_state_dict(ck["model_state_dict"]); TRY_DEV = "cuda" if torch.cuda.is_available() else "cpu"
tmodel = tmodel.to(TRY_DEV).eval()
T_IDX = [ck["class_names"].index(c) for c in ck["cheating_classes"]]
THR = TRY_THRESHOLD or float(ck.get("threshold", 0.5))
TEMP_INFER = float(ck.get("temperature", 1.0))
IN_SIZE = int(ck.get("input_size", 224))
tyolo = YOLO(CFG["YOLO_WEIGHTS"])

@torch.no_grad()
def score_crops(crops_bgr):
    if not crops_bgr: return np.zeros(0)
    x = torch.stack([
        TF.normalize(TF.resize(torch.from_numpy(np.ascontiguousarray(c[:, :, ::-1])).permute(2, 0, 1).float() / 255,
                               [IN_SIZE, IN_SIZE], antialias=True), MEAN, STD)
        for c in crops_bgr
    ])
    p = torch.softmax(tmodel(x.to(TRY_DEV)), 1)[:, T_IDX].sum(1).cpu().numpy()
    return apply_temperature(p, TEMP_INFER)

import matplotlib.pyplot as plt

def draw(img, box, text, flag):
    x1, y1, x2, y2 = map(int, box); col = (0, 0, 255) if flag else (0, 200, 0); fs = max(0.5, img.shape[1] / 1100)
    cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, 2); y = max(th + 6, y1)
    cv2.rectangle(img, (x1, y - th - 6), (x1 + tw + 4, y), col, -1)
    cv2.putText(img, text, (x1 + 2, y - 4), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), 2)

def test_image(path):
    img = cv2.imread(path); H, W = img.shape[:2]
    b = tyolo.predict(img, classes=[0], conf=0.3, verbose=False)[0].boxes.xyxy.cpu().numpy()
    if len(b) == 0 or (len(b) == 1 and (b[0, 2] - b[0, 0]) * (b[0, 3] - b[0, 1]) / (W * H) >= 0.6):
        b = np.array([[0, 0, W - 1, H - 1]], float)
    s = score_crops([img[int(y1):int(y2), int(x1):int(x2)] for x1, y1, x2, y2 in b])
    out = img.copy()
    for i, (box, sc) in enumerate(zip(b, s)):
        draw(out, box, f"P{i+1} {sc:.2f}" + (" FLAG" if sc >= THR else ""), sc >= THR)
        print(f"person {i+1}: {sc:.2f}" + (" FLAG" if sc >= THR else ""))
    plt.figure(figsize=(10, 7)); plt.imshow(out[:, :, ::-1]); plt.axis("off"); plt.show()
    cv2.imwrite(str(TRY_OUT / (Path(path).stem + "_GroupProject.jpg")), out)

print("loaded", TRY_MODEL, "| thr", round(THR, 3), "| T", TEMP_INFER)
if SMOKE_TEST and len(train_all):
    test_image(train_all.crop.iloc[0])
    print("smoke try-it OK")


In [ ]:
print('Initialization of the execution pipeline')

In [ ]:
import io, random, math, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision import models
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix, precision_recall_curve
from PIL import Image

print('Assembling and running entire training pipeline...')

# --- 1. Augmentations & MixUp ---
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
class DownUp:
    def __call__(self, im):
        if random.random() > 0.25: return im
        s = random.randint(max(16, CFG["IMG"] // 4), max(17, CFG["IMG"] // 2))
        return im.resize((s, s), Image.BILINEAR).resize(im.size, Image.BILINEAR)
class Jpeg:
    def __call__(self, im):
        if random.random() > 0.3: return im
        b = io.BytesIO(); im.save(b, "JPEG", quality=random.randint(28, 92)); b.seek(0)
        return Image.open(b).convert("RGB")
train_tf = T.Compose([
    T.RandomResizedCrop(CFG["IMG"], scale=(0.55, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.RandomRotation(12)], p=0.5),
    T.ColorJitter(0.35, 0.35, 0.25, 0.04),
    T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.25),
    T.RandomAutocontrast(p=0.2),
    DownUp(), Jpeg(),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((CFG["IMG"], CFG["IMG"])), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf_flip = T.Compose([T.Resize((CFG["IMG"], CFG["IMG"])), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(MEAN, STD)])

def mixup_batch(x, y, alpha):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

# --- 2. Dataset & Loaders ---
class Crops(Dataset):
    def __init__(self, d, tf, soft=False):
        self.p = d.crop.tolist()
        self.t = (d.y.tolist() if soft else d.t.tolist())
        self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), self.t[i]

NW = 0
def train_loader(d):
    ds = Crops(d, train_tf, soft=False)
    if CFG["LOSS"] == "sampler":
        w = 1.0 / d.t.map(d.t.value_counts()).values
        return DataLoader(ds, CFG["BATCH"], sampler=WeightedRandomSampler(w, len(d), True), drop_last=True, num_workers=NW)
    return DataLoader(ds, CFG["BATCH"], shuffle=True, drop_last=True, num_workers=NW)

@torch.no_grad()
def _predict_tf(model, d, tf):
    model.eval(); out = []
    for x, _ in DataLoader(Crops(d, tf), CFG["BATCH"], num_workers=NW):
        n = len(x)
        if n < CFG["BATCH"]: x = torch.cat([x, x.new_zeros(CFG["BATCH"] - n, *x.shape[1:])])
        with autocast():
            p = torch.softmax(model(x.to(DEVICE)).float(), 1)[:, CHEAT_IDX].sum(1)
        sync(); out.append(p.cpu()[:n])
    return torch.cat(out).numpy()

def predict(model, d, tta=None):
    use_tta = CFG["USE_TTA"] if tta is None else tta
    p0 = _predict_tf(model, d, eval_tf)
    if not use_tta: return p0
    p1 = _predict_tf(model, d, eval_tf_flip)
    return 0.5 * (p0 + p1)

# --- 3. Model Architecture & EMA ---
def build_model(n, arch=None):
    arch = arch or CFG["ARCH"]
    try:
        if arch ==

In [ ]:
# Execute all the prerequisite functions, configurations, and data preparation steps
import os, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import json
import math

# Define default configuration first
CFG = dict(
    AUTHOR="GroupProject",
    SOURCE="synthetic",
    DRIVE_ROOT="/tmp/GroupProject_smoke/data",
    SOURCES=[
        dict(name="examdatasetnew", type="images", path="examdatasetnew"),
        dict(name="ExamCheatingDataset", type="images", path="ExamCheatingDataset"),
    ],
    OUT_DIR="/tmp/GroupProject_smoke/out",
    WORK_DIR="/tmp/GroupProject_smoke/work",
    NORMAL_CLASSES=["normal", "normal act", "not cheating", "notcheating"],
    FRONTWATCHING_SOFT=0.35,
    CROSS_PAIRS=[("examdatasetnew", "ExamCheatingDataset")],
    ARCH="efficientnet_b0",
    ARCH_FALLBACK="efficientnet_b0",
    HEAD="binary",
    IMG=64,
    CACHE_SIZE=72,
    BATCH=8, EPOCHS_HEAD=1, EPOCHS_FT=2, LR_HEAD=1e-3, LR_FT=8e-5, WD=1e-4,
    WARMUP_EPOCHS=0, GRAD_CLIP=1.0,
    PATIENCE=1, LOSS="weighted_ce",
    BF16=False,
    N_FOLDS=3, FOLDS_TO_RUN=2,
    TARGET_PRECISION=0.5,
    EMA_DECAY=0.995,
    MIXUP_ALPHA=0.1,
    USE_TTA=True,
    EXPORT_ENSEMBLE=True,
    DROP_FLIP_COPY_FILES=True, DEDUP_HAMMING=4,
    YOLO_WEIGHTS="yolov8n.pt", YOLO_CONF=0.3, CROP_IF_PERSON_AREA_BELOW=0.6, CROP_PAD=0.12, MIN_CROP_PX=40,
    SEED=42,
)

SMOKE_TEST = True
SOURCES = [s for s in CFG["SOURCES"] if s.get("enabled", True)]
print("Configuration and environment successfully initialized!")

In [ ]:
# Install opencv if not present, then initialize synthetic data
import subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)
    import cv2

import numpy as np
from PIL import Image
from pathlib import Path
try:
    from ultralytics.utils import ASSETS
    people = [np.array(Image.open(ASSETS / f).convert("RGB")) for f in ("zidane.jpg", "bus.jpg")]
except Exception:
    people = [(np.random.rand(120, 90, 3) * 255).astype("uint8") for _ in range(2)]
rng = np.random.default_rng(0)
layout = {"examdatasetnew": ["backwatching", "frontwatching", "normal", "showinggestures", "sidewatching", "suspiciouswatching"],
          "ExamCheatingDataset": ["normal act", "looking friend", "giving code", "giving object", "cheating"]}
for ds, classes in layout.items():
    for ci, c in enumerate(classes):
        d = Path(CFG["DRIVE_ROOT"]) / ds / c; d.mkdir(parents=True, exist_ok=True)
        for i in range(14):
            a = (rng.random((96, 80, 3)) * 60).astype("uint8"); a[:, :, ci % 3] += 120 + 10 * ci
            Image.fromarray(a).save(d / f"{i}.jpg")
        Image.fromarray(a).save(d / f"{i}_f.jpg")
        for k in range(2):
            b = people[k % 2].astype(int) + rng.integers(-15, 15, people[k % 2].shape)
            Image.fromarray(np.clip(b, 0, 255).astype("uint8")).save(d / f"group_{k}.jpg")
print("synthetic data ready")

In [ ]:
# Re-execute image parsing for synthetic dataset with inline helper definitions
import re, numpy as np, pandas as pd, cv2
from pathlib import Path

IMG_EXT, VID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}, {".mp4", ".avi", ".mov", ".mkv", ".webm"}
norm = lambda s: re.sub(r"[_\-]+", " ", str(s).strip().lower())

def hard_cheating(cls):
    c = norm(cls)
    if c in [norm(x) for x in CFG["NORMAL_CLASSES"]]: return 0
    if c == "frontwatching": return None
    return 1

def label_of(s, cls, rel):
    lm = {norm(k): v for k, v in (s.get("label_map") or {}).items()}
    if lm:
        if norm(cls) in lm: return float(lm[norm(cls)])
        hit = next((v for k, v in lm.items() if k in norm(rel)), None)
        return None if hit is None else float(hit)
    if norm(cls) == "frontwatching":
        return float(CFG["FRONTWATCHING_SOFT"])
    y = hard_cheating(cls)
    return None if y is None else float(y)

def src_root(s):
    p = Path(s["path"]) if Path(s["path"]).is_absolute() else Path(CFG["DRIVE_ROOT"]) / s["path"]
    return p

rows, skipped = [], []
for s in SOURCES:
    root = src_root(s)
    if not root.exists(): print("MISSING:", s["name"], root); continue
    kind, multi = s.get("type", "images"), s.get("multi", "all" if s.get("type") == "video" else "largest")
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root); cls = norm(rel.parts[0]) if len(rel.parts) > 1 else norm(p.stem)
        if kind == "images" and p.suffix.lower() in IMG_EXT:
            y = label_of(s, cls, rel)
            if y is None: skipped.append(str(rel)); continue
            rows.append(dict(path=str(p), dataset=s["name"], kind="images", cls=cls, y=y, y_hard=int(y >= 0.5),
                             subject=None, multi=multi, train=s.get("train", True)))

df = pd.DataFrame(rows)
print(df.groupby(["dataset", "cls"]).agg(n=("y", "size"), y_mean=("y", "mean")).to_string())

## Notes (GroupProject)
- Run on Colab with the shared Drive folder mounted; outputs go to `SmartInvigilation_runs/GroupProject_effnetb3/`.
- Submit **`GroupProject.ipynb`** plus exported `GroupProject_best.pt` / `metrics_GroupProject.json`.
- Report **precision, recall, macro-F1, AP** (and false-alarm rate on video), not accuracy.
- Datasets: CUI-EXAM / IEEE Access 2021; ExamCheatingDataset (Kaggle); Hashemite if enabled.
- **Declare AI assistance** if required by the module.
